# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. 
2. 
3. 
4. 
5. 

In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [3]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [ ]:
# База

In [4]:
def to_num(s):
    if not s: return 0.0
    s_clean = ''.join(c for c in str(s).replace(',', '.') if c.isdigit() or c == '.')
    return float(s_clean) if s_clean else 0.0

def clean_city(s):
    city_map = {
        'almaty': 'Алматы', 'astana': 'Астана', 
        'shymkent': 'Шымкент', 'atyrau': 'Атырау',
        'aktau': 'Актау', 'karaganda': 'Караганда'
    }
    cleaned = str(s).strip().lower()
    return city_map.get(cleaned, cleaned.capitalize())

category_counts = {}
for r in rows:
    cat = r.get('category', 'Неизвестно')
    category_counts[cat] = category_counts.get(cat, 0) + 1

sorted_categories = sorted(category_counts.items(), key=lambda x: x[1], reverse=True)
print("2. Заказы по категориям:", sorted_categories)
cat_revenue = {}
cat_paid_counts = {}
for r in rows:
    if r.get('status', '').lower() in ['paid', 'оплачен', 'true', '1', 'оплачено']:
        cat = r.get('category', 'Неизвестно')
        amt = to_num(r.get('amount', r.get('price', 0)))
        cat_revenue[cat] = cat_revenue.get(cat, 0) + amt
        cat_paid_counts[cat] = cat_paid_counts.get(cat, 0) + 1

print("\n3. Таблица по категориям:")
print(f"{'Категория':<20} | {'Выручка':<15} | {'Средний чек':<15}")
for cat in cat_revenue:
    rev = cat_revenue[cat]
    cnt = cat_paid_counts[cat]
    avg = rev / cnt if cnt > 0 else 0
    print(f"{cat:<20} | {rev:<15.2f} | {avg:<15.2f}")

month_counts = {}
for r in rows:
    month = r.get('order_date', '')[:7]
    if month:
        month_counts[month] = month_counts.get(month, 0) + 1
print("\n4. Заказы по месяцам:", month_counts)

2. Заказы по категориям: [('Одежда', 1529), ('Электроника', 1185), ('Дом', 1105), ('Красота', 991), ('Книги', 653)]

3. Таблица по категориям:
Категория            | Выручка         | Средний чек    
Одежда               | 18321080.00     | 13285.77       
Красота              | 5507620.00      | 6188.34        
Дом                  | 10516320.00     | 10654.83       
Книги                | 2294100.00      | 3934.99        
Электроника          | 65270240.00     | 61286.61       

4. Заказы по месяцам: {'2026-09': 2782, '2026-10': 2681}


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [ ]:
# Стандарт

## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [ ]:
# Челлендж